In [1]:
# NORTHSTAR -- Tower 34 Schedule Automation QA for Solace Browser
# DNA: schedule(auto) = create(cron) x approve(first) x run(recurring) x budget(enforce) x notify(result) x pause(control) x evidence(log)
# Probes schedule creation, first-run approval, recurring execution, budget, notifications
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "schedule-t34-qa"
NOTEBOOK_PATH = "notebooks/qa/20-schedule-t34-qa.ipynb"
TOWER = 34
TOWER_NAME = "Tower of Schedule Automation"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"

results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

Tower 34: Tower of Schedule Automation


In [2]:
# F1 -- Omar Hassan: Schedule Creation (Marketing Manager)
# Q: Can I schedule a recipe daily at 7am? Timezone-aware?

# Probe: schedule page exists in web UI
schedule_html = read_file(WEB / "schedule.html")
has_schedule_page = len(schedule_html) > 200
record("F1-P1", "Schedule page exists in web UI", has_schedule_page,
       f"{len(schedule_html)} bytes")

# Probe: schedule-core.js handles schedule logic
schedule_core = read_file(WEB / "js" / "schedule-core.js")
has_core = bool(re.search(r'schedule|cron|recur|repeat|interval', schedule_core, re.IGNORECASE))
record("F1-P2", "schedule-core.js handles scheduling logic", has_core,
       f"{len(schedule_core)} bytes")

# Probe: schedule handles timezone
has_timezone = bool(re.search(r'timezone|tz|utc|offset|iana|intl.*timezone', schedule_core, re.IGNORECASE))
record("F1-P3", "Schedule handles timezone awareness", has_timezone)

# Probe: schedule-calendar.js for calendar view
schedule_cal = read_file(WEB / "js" / "schedule-calendar.js")
has_calendar = bool(re.search(r'calendar|month|week|day|grid|view', schedule_cal, re.IGNORECASE))
record("F1-P4", "Schedule calendar view exists", has_calendar,
       f"{len(schedule_cal)} bytes")

PASS: Schedule page exists in web UI -- 16783 bytes
PASS: schedule-core.js handles scheduling logic -- 24856 bytes
PASS: Schedule handles timezone awareness
PASS: Schedule calendar view exists -- 10709 bytes


In [3]:
# F2 -- Lisa Chen: First-Run Approval (VP Customer Success)
# Q: Does system require approval before first automated run?

# Probe: schedule-approvals.js handles first-run approval
schedule_approvals = read_file(WEB / "js" / "schedule-approvals.js")
has_approval = bool(re.search(r'approv|confirm|first.*run|preview|dry.*run', schedule_approvals, re.IGNORECASE))
record("F2-P1", "Schedule approvals JS handles first-run approval", has_approval,
       f"{len(schedule_approvals)} bytes")

# Probe: approvals gate module in Python
approval_gate = read_file(SRC / "approvals" / "gate.py")
has_gate = bool(re.search(r'approv|gate|require|pending|status', approval_gate, re.IGNORECASE))
record("F2-P2", "Approval gate Python module exists", has_gate,
       f"{len(approval_gate)} bytes")

# Probe: approval test exists
approval_test = TESTS / "test_approvals.py"
has_approval_test = approval_test.exists() and approval_test.stat().st_size > 100
record("F2-P3", "Approvals test exists", has_approval_test)

# Probe: elevated approvals for high-risk schedules
elevated = read_file(SRC / "approvals" / "elevated.py")
has_elevated = len(elevated) > 100
record("F2-P4", "Elevated approvals module exists for high-risk ops", has_elevated)

PASS: Schedule approvals JS handles first-run approval -- 14234 bytes
PASS: Approval gate Python module exists -- 20873 bytes
PASS: Approvals test exists
PASS: Elevated approvals module exists for high-risk ops


In [4]:
# F3 -- Recurring Execution + Budget Enforcement
# Q: Does schedule enforce budget per run? Can it pause on limit?

# Probe: schedule-evidence.js captures run evidence
schedule_evidence = read_file(WEB / "js" / "schedule-evidence.js")
has_evidence = bool(re.search(r'evidence|capture|record|log|hash', schedule_evidence, re.IGNORECASE))
record("F3-P1", "Schedule evidence JS captures run evidence", has_evidence,
       f"{len(schedule_evidence)} bytes")

# Probe: budget gates can enforce per-schedule limits
budget_code = read_file(SRC / "budget_gates.py")
has_budget = bool(re.search(r'budget|cost|limit|cap', budget_code, re.IGNORECASE))
record("F3-P2", "Budget gates can enforce schedule cost limits", has_budget)

# Probe: schedule operations test exists
schedule_test = TESTS / "test_schedule_operations.py"
has_sched_test = schedule_test.exists() and schedule_test.stat().st_size > 100
record("F3-P3", "Schedule operations test exists", has_sched_test)

PASS: Schedule evidence JS captures run evidence -- 21310 bytes
PASS: Budget gates can enforce schedule cost limits
PASS: Schedule operations test exists


In [5]:
# F4-F5 -- Notifications + Pause/Resume
# Q: Do I get notified when a scheduled run completes?
# Q: Can I pause/resume a schedule?

# Probe: schedule core has pause/resume/cancel
schedule_core = read_file(WEB / "js" / "schedule-core.js")
has_pause = bool(re.search(r'pause|resume|cancel|stop|disable|enable', schedule_core, re.IGNORECASE))
record("F4-P1", "Schedule supports pause/resume/cancel", has_pause)

# Probe: schedule has status tracking (pending, running, completed, failed)
has_status = bool(re.search(r'status|pending|running|completed|failed|success', schedule_core, re.IGNORECASE))
record("F4-P2", "Schedule tracks run status", has_status)

# Probe: push alerts for schedule notifications
push_alerts = read_file(SRC / "yinyang" / "push_alerts.py")
has_push = bool(re.search(r'push|notify|alert|schedule|complete', push_alerts, re.IGNORECASE))
record("F4-P3", "Push alerts module can notify on schedule completion", has_push)

PASS: Schedule supports pause/resume/cancel
PASS: Schedule tracks run status
PASS: Push alerts module can notify on schedule completion


In [6]:
# F6-F7 -- Evidence Logging + NEGATIVE SPACE
# Q: Does each schedule run produce an evidence record?

# Probe: schedule HTML page has data-i18n attributes
schedule_html = read_file(WEB / "schedule.html")
has_i18n = bool(re.search(r'data-i18n|lang=|i18n', schedule_html, re.IGNORECASE))
record("F6-P1", "Schedule page has i18n support", has_i18n)

# Probe: schedule CSS exists
schedule_css = read_file(WEB / "css" / "schedule.css")
has_css = len(schedule_css) > 100
record("F6-P2", "Schedule CSS exists", has_css,
       f"{len(schedule_css)} bytes")

# NEGATIVE SPACE: schedule-old.js should not be loaded in production
old_js = WEB / "js" / "schedule-old.js"
old_js_exists = old_js.exists()
# Check if it's referenced from schedule.html
old_referenced = bool(re.search(r'schedule-old\.js', schedule_html))
record("NS-P1", "schedule-old.js not loaded in production HTML",
       not old_referenced,
       f"old.js exists={old_js_exists}, referenced={old_referenced}")

# NEGATIVE SPACE: no bare excepts in schedule JS
schedule_js_files = [WEB / "js" / f for f in ["schedule-core.js", "schedule-calendar.js", "schedule-approvals.js", "schedule-evidence.js"]]
uncaught = 0
for sjf in schedule_js_files:
    if sjf.exists():
        content = sjf.read_text(encoding='utf-8', errors='replace')
        # Check for catch blocks that ignore error
        uncaught += len(re.findall(r'catch\s*\([^)]*\)\s*\{\s*\}', content))
record("NS-P2", "No empty catch blocks in schedule JS",
       uncaught == 0, f"{uncaught} empty catch blocks")

PASS: Schedule page has i18n support
PASS: Schedule CSS exists -- 28615 bytes
PASS: schedule-old.js not loaded in production HTML -- old.js exists=True, referenced=False
PASS: No empty catch blocks in schedule JS -- 0 empty catch blocks


In [7]:
# EVIDENCE SUMMARY -- Tower 34 Schedule Automation QA
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"

summary = {
    "surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_probes": total, "passed": passed, "failed": failed,
    "score": score, "grade": grade, "finding_rate": finding_rate,
    "converged": finding_rate < 5,
    "findings": [r for r in results if r["status"] == "FAIL"],
    "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]
}

print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))

TOWER 34: Tower of Schedule Automation
Probes: 18 | Passed: 18 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: 3458fc22c0c7d0c3

{
  "surface": "schedule-t34-qa",
  "tower": 34,
  "tower_name": "Tower of Schedule Automation",
  "timestamp": "2026-03-06T10:26:42.454553",
  "total_probes": 18,
  "passed": 18,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "3458fc22c0c7d0c3"
}
